# Week 4: WideResNet-50 on MVTec AD

This runs the same 15 categories and three seeds as the ResNet-18 experiment, with 500 epochs per run. The reference WideResNet settings also change the flow kernels and pretrained weights; this is a comparison of the two reference configurations.

Choose **Runtime > Change runtime type > GPU**. Use `week4_study.zip` with this notebook. Keep at least **25 GB free in Google Drive** for the dataset, final weights, maps, and checkpoints. Check Drive's storage page first; the notebook cannot reliably infer your Drive quota from runtime disk space. Do not run two copies against the same study folder.

First do steps 1–4. Then disconnect and delete the runtime, reconnect to the same GPU type, repeat steps 1–3, and run step 5. This checks recovery using a fresh runtime before the study starts. Do not use Run all for this first test.

The included bottle images and downloaded MVTec AD dataset use [CC BY-NC-SA 4.0](https://www.mvtec.com/research-teaching/datasets/mvtec-ad).


## 1. Connect Drive

Approve the Drive access prompt. This notebook stores its files under `MyDrive/CISC473/week4_wrn50`. On later sessions, run this cell again.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, zipfile

drive.mount('/content/drive')
store = Path('/content/drive/MyDrive/CISC473/week4_wrn50')
store.mkdir(parents=True, exist_ok=True)
project = Path('/content/week4-study')
project.mkdir(exist_ok=True)
boot_id = Path('/proc/sys/kernel/random/boot_id').read_text().strip()
subprocess.run(['nvidia-smi'], check=True)

def sha256(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def unpack(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as z:
        for item in z.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Invalid archive path')
        z.extractall(destination)

def command(args, cwd=project):
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    with subprocess.Popen([str(x) for x in args], cwd=cwd, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
        code = p.wait()
    if code:
        raise RuntimeError(f'Command exited with code {code}. See the output above.')


## 2. Load the project

On the first session, choose `week4_study.zip` when prompted. A checked copy is kept in Drive, so you will not need to upload it after reconnecting.


In [ ]:
expected_package = '2b3dfda16119ee49d94244822f99662dc182d8892e2dc3ba77dc331e5f58edc4'
saved_package = store / 'week4_study.zip'
local_package = Path('/content/week4_study.zip')
if not saved_package.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload only week4_study.zip')
    name, payload = next(iter(uploaded.items()))
    if hashlib.sha256(payload).hexdigest() != expected_package:
        raise ValueError('This package does not match the notebook.')
    local_package.write_bytes(payload)
    del uploaded, payload
    temporary = saved_package.with_suffix('.partial')
    shutil.copyfile(local_package, temporary)
    if sha256(temporary) != expected_package:
        raise IOError('Drive package copy failed')
    temporary.replace(saved_package)
else:
    shutil.copyfile(saved_package, local_package)
if sha256(local_package) != expected_package:
    raise ValueError('Stored package checksum does not match.')
unpack(local_package, project)
manifest = json.loads((project / 'bundle_manifest.json').read_text())
for name, expected in manifest.items():
    if sha256(project / name) != expected:
        raise ValueError(f'File checksum mismatch: {name}')
print('Project checked.')


## 3. Install the environment

Run this again on each new runtime. It restores the same pinned environment. The first installation takes several minutes.


In [ ]:
if shutil.disk_usage('/content').free < 20 * 1024**3:
    raise RuntimeError('Need at least 20 GiB free on the Colab runtime.')
command([sys.executable, '-m', 'pip', 'install', 'uv'])
os.environ['PATH'] = str(Path.home() / '.local/bin') + os.pathsep + os.environ['PATH']
command([sys.executable, 'scripts/setup.py', '--device', 'cuda'])
python = project / '.venv/bin/python'
command([python, '-c', 'import torch; print(torch.cuda.get_device_name(0))'])

def run_study(mode):
    command([python, '-u', 'scripts/week4_colab.py', '--mode', mode, '--store', store])


## 4. Prepare the recovery test (first session only)

This trains bottle for one epoch, saves the checkpoint to Drive, then trains a second epoch and records its exact state for comparison. It is a test, separate from the final results.

After this finishes, choose **Runtime > Disconnect and delete runtime**. Reconnect with the same GPU type, repeat steps 1–3, skip this step, and run step 5.


In [ ]:
origin = store / 'recovery/original_runtime.json'
if not origin.exists():
    run_study('recovery-save')
    origin.write_text(json.dumps({'boot_id': boot_id}, indent=2))
else:
    print('Recovery checkpoint already prepared. Continue in a fresh runtime with step 5.')


## 5. Check recovery (fresh runtime)

This restores epoch 1 from Drive and repeats epoch 2. The model, optimizer, random states, and loss must match the uninterrupted run exactly. If this fails, stop and keep the error output. The full study remains disabled until this check passes.


In [ ]:
origin = json.loads((store / 'recovery/original_runtime.json').read_text())
if origin['boot_id'] == boot_id:
    raise RuntimeError('Use a fresh runtime: disconnect and delete, reconnect, then repeat steps 1–3.')
run_study('recovery-check')
(store / 'recovery/fresh_runtime_passed.json').write_text(json.dumps({'boot_id': boot_id}))


## 6. Load the full dataset

Only continue after the recovery check passes. On the first study session, this downloads the official dataset, verifies its archive checksum, and saves a reusable archive in Drive. Later sessions restore that archive to the runtime. Training reads local images, so it does not make thousands of small reads from Drive.


In [ ]:
if not (store / 'recovery/fresh_runtime_passed.json').exists():
    raise RuntimeError('Finish the fresh-runtime recovery test first.')
data = project / 'data/mvtec_ad'
saved_data = store / 'mvtec_ad.zip'
data_receipt = store / 'mvtec_ad_zip.json'
local_data = Path('/content/mvtec_ad.zip')
if data_receipt.exists():
    receipt = json.loads(data_receipt.read_text())
    if not (data / 'download.json').exists():
        shutil.copyfile(saved_data, local_data)
        if sha256(local_data) != receipt['sha256']:
            raise ValueError('Dataset archive checksum mismatch.')
        unpack(local_data, data)
        local_data.unlink()
else:
    command([python, 'scripts/download_data.py', '--root', data])
    with zipfile.ZipFile(local_data, 'w', zipfile.ZIP_STORED) as z:
        for path in sorted(data.rglob('*')):
            if path.is_file():
                z.write(path, path.relative_to(data).as_posix())
    expected = sha256(local_data)
    temporary = saved_data.with_suffix('.partial')
    shutil.copyfile(local_data, temporary)
    if sha256(temporary) != expected:
        raise IOError('Drive dataset copy failed.')
    temporary.replace(saved_data)
    data_receipt.write_text(json.dumps({'sha256': expected}, indent=2))
    local_data.unlink()
print('Full dataset ready.')


## 7. Start or resume the study

Change `START_STUDY` to `True` when ready. This runs all remaining categories and seeds automatically while the session lasts. Checkpoints are saved to Drive every 25 epochs. After a disconnect, up to 24 completed epochs plus an interrupted epoch may need to be repeated.

For later sessions: reconnect to a GPU, run steps **1, 2, 3, 6, and 7**. Completed runs are checked and skipped; the unfinished run resumes from its latest readable checkpoint. You do not need to repeat the recovery test after it has passed. Colab still controls session duration and GPU availability; this notebook cannot reconnect itself.

Results appear in `MyDrive/CISC473/week4_wrn50/runs/<category>/seed_<number>/`. Each finished run has metrics, image scores, anomaly maps, heatmap previews, training history, and final trainable weights. The frozen backbone is identified by its checksum in the settings and downloaded again when needed. A `complete.json` receipt is written only after all result copies are checked.


In [ ]:
START_STUDY = False
if not START_STUDY:
    print('Set START_STUDY = True when ready to start or resume.')
else:
    if not (store / 'recovery/fresh_runtime_passed.json').exists():
        raise RuntimeError('Finish the fresh-runtime recovery test first.')
    run_study('study')


## 8. Check progress

Run this when the training cell is stopped or after reconnecting. Saved-epoch progress may lag the live training output by up to 24 completed epochs.


In [ ]:
completed = sorted((store / 'runs').glob('*/seed_*/complete.json'))
print(f'Completed runs: {len(completed)}/45')
for progress in sorted((store / 'runs').glob('*/seed_*/progress.json')):
    if not (progress.parent / 'complete.json').exists():
        print(json.loads(progress.read_text()))
